# A-Maze-ing

## Read the configuration file

Just open the file, split the options turn them into a dictionary.

In [1]:
with open("config.txt") as f:
    txt = f.read()
    opts = txt.split("\n")
    tup = [tuple(i.split("=")) for i in opts]
    conf = {i[0]: i[1] for i in tup}
print(conf)

{'HEIGHT': '17', 'WIDTH': '31', 'ENTRY': '0,0', 'EXIT': '16,30', 'OUTPUT_FILE': 'maze.txt', 'PERFECT': 'True'}


## Create a $h \times w$ array

Two different arrays:

* The maze itself.
* The visited cells.

The 42 was hardcoded.

In [2]:
from numpy import zeros, ones, mean, array


size = (int(conf["HEIGHT"]), int(conf["WIDTH"]))
maze = ones(size, int) * 15
visited = zeros(size, int)
perfect = bool(conf["PERFECT"])
if visited.shape[0] < 7 and visited.shape[1] < 9:
    is_42 = 0
    print("The '42' cannot be printed in a maze this size.")
else:
    is_42 = 1
    hs, ws = visited.shape[0] // 2 - 2, visited.shape[1] // 2 - 3
    pos_42 = (
        (hs, ws), (hs + 1, ws), (hs + 2, ws), (hs + 2, ws + 1),
        (hs + 2, ws + 2), (hs + 3, ws + 2), (hs + 4, ws + 2),
        (hs, ws + 4), (hs, ws + 5), (hs, ws + 6), (hs + 1, ws + 6),
        (hs + 2, ws + 6), (hs + 2, ws + 5), (hs + 2, ws + 4),
        (hs + 3, ws + 4), (hs + 4, ws + 4), (hs + 4, ws + 5),
        (hs + 4, ws + 6)
    )
    for i in pos_42:
        visited[i[0]][i[1]] = -1
    """lab_42 = [
        (hs + 1, ws + 3),
        (hs + 1, ws + 4),
        (hs + 1, ws + 5),
        (hs + 3, ws + 5),
        (hs + 3, ws + 6),
        (hs + 3, ws + 7),
    ]
    if perfect is False:
        lab_42.extend(
            [
                (hs + 2, ws + 3),
                (hs + 3, ws + 3),
                (hs + 4, ws + 3),
                (hs + 5, ws + 3)
            ]
        )
    for i in lab_42:
        visited[i[0], i[1]] = 1"""
print(visited)
"""
maze[hs + 1][ws + 3] = 13
maze[hs + 1][ws + 4] = 5
maze[hs + 1][ws + 5] = 7
maze[hs + 3][ws + 5] = 13
maze[hs + 3][ws + 6] = 5
maze[hs + 3][ws + 7] = 7
if perfect is False:
    maze[hs + 1][ws + 3] = 8
    maze[hs + 2][ws + 3] = 10
    maze[hs + 3][ws + 3] = 10
    maze[hs + 4][ws + 3] = 10
    maze[hs + 5][ws + 3] = 14
"""
for i in maze:
    print(i)

[[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0 -1  0  0  0 -1 -1 -1  0  0  0  0  0
   0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0 -1  0  0  0  0  0 -1  0  0  0  0  0
   0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0 -1 -1 -1  0 -1 -1 -1  0  0  0  0  0
   0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0 -1  0 -1  0  0  0  0  0  0  0
   0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0 

## Manage errors

Checks:

* Maze dimensions must be positive.
* Maze entrance and exit are inside the maze.
* Maze entrance and exit are not inside forbidden tiles.
* All maze walls in a tile are coherent with the neighbouring tiles.

In [3]:
from numpy.typing import NDArray


class MazeError(Exception):
    def __init__(self, msg: str):
        self.msg = msg


start = conf["ENTRY"].split(",")
end = conf["EXIT"].split(",")
entr = (int(start[0]), int(start[1]))
exit = (int(end[0]), int(end[1]))

if entr == exit:
    raise MazeError("Maze entrance and exit must be different")

if size[0] <= 0 or size[1] <= 0:
    raise MazeError("Maze dimensions must be positive")

in_bounds = [
    entr[0] >= 0,
    entr[1] >= 0,
    entr[0] < size[0],
    entr[1] < size[1],
    exit[0] >= 0,
    exit[1] >= 0,
    exit[0] < size[0],
    exit[1] < size[1]
]
if False in in_bounds:
    raise MazeError("Entrance or exit of the maze seem to be out of bounds")

if visited.shape[0] > 6 or visited.shape[1] > 8:
    in_42 = []
    in_42.extend([True for i in pos_42 if i[0] == entr[0] and i[1] == entr[1]])
    in_42.extend([True for i in pos_42 if i[0] == exit[0] and i[1] == exit[1]])
    if True in in_42:
        raise MazeError("Entrance or exit of the maze seem to be inside the 42")


def get_walls(n: int) -> tuple[int]:
    if n > 15 or n < 0:
        raise ValueError
    bits = []
    bits.append(n // 8)
    n %= 8
    bits.append(n // 4)
    n %= 4
    bits.append(n // 2)
    n %= 2
    bits.append(n)
    bits.reverse()
    return tuple(bits)


def check_tile(maze: NDArray, coords: tuple[int]):
    i, j = coords
    tile = int(maze[i, j])
    msg = f"Maze tile {coords} has incoherent walls"
    walls = get_walls(tile)
    if i == 0:
        if walls[0] == 0:
            raise MazeError(f"{msg} with north border")
    else:
        if walls[0] != get_walls(maze[i - 1][j])[2]:
            raise MazeError(f"{msg} with tile {(i - 1, j)}")
    if i == maze.shape[0] - 1:
        if walls[2] == 0:
            raise MazeError(f"{msg} with south border")
    else:
        if walls[2] != get_walls(maze[i + 1][j])[0]:
            raise MazeError(f"{msg} with tile {(i + 1, j)}")
    if j == 0:
        if walls[3] == 0:
            raise MazeError(f"{msg} with west border")
    else:
        if walls[3] != get_walls(maze[i][j - 1])[1]:
            raise MazeError(f"{msg} with tile {(i, j - 1)}")
    if j == maze.shape[1] - 1:
        if walls[1] == 0:
            raise MazeError(f"{msg} with east border")
    else:
        if walls[1] != get_walls(maze[i][j + 1])[3]:
            raise MazeError(f"{msg} with tile {(i, j + 1)}")

## Maze generator

Gonna be a Wilson algorithm.

In [4]:
import random as rng


def manhattan(curr: tuple[int], neigh: tuple[int]) -> int:
    return abs(curr[0] - neigh[0]) + abs(curr[1] - neigh[1])



def connect(maze: NDArray, curr: tuple[int], neigh: tuple[int]) -> None:
    if manhattan(curr, neigh) != 1:
        return
        
    if curr[0] == neigh[0] + 1:
        maze[curr[0]][curr[1]] -= 1
        maze[neigh[0]][neigh[1]] -= 4
    if curr[0] == neigh[0] - 1:
        maze[curr[0]][curr[1]] -= 4
        maze[neigh[0]][neigh[1]] -= 1
    if curr[1] == neigh[1] - 1:
        maze[curr[0]][curr[1]] -= 2
        maze[neigh[0]][neigh[1]] -= 8
    if curr[1] == neigh[1] + 1:
        maze[curr[0]][curr[1]] -= 8
        maze[neigh[0]][neigh[1]] -= 2


def direction() -> tuple[str, int]:
    if rng.random() < .5:
        axis = "V"
    else:
        axis = "H"
    if rng.random() < .5:
        step = -1
    else:
        step = 1
    return (axis, step)


def count_visited(visited: NDArray) -> int:
    n = 0
    for i in range(visited.shape[0]):
        for j in range(visited.shape[1]):
            if visited[i][j] == 1:
                n += 1
    return n


def wilson_erase(visited: NDArray, path: list[tuple], point: tuple[int]):
    while path[-1] != point:
        visited[path[-1][:]] = 0
        path.pop()


# Initialization
current = [rng.randrange(0, size[0] - 1), rng.randrange(0, size[1] - 1)]
visited[current[0]][current[1]] = 1
# Alodus-Broder Algorithm
i = 0
"""while count_visited(visited) < (maze.shape[0] * maze.shape[1] - 18 * is_42) // 3:
    direct = direction()
    try:
        if direct[0] == "V":
            neighbour = [current[0] + direct[1], current[1]]
        else:
            neighbour = [current[0], current[1] + direct[1]]
        if visited[neighbour[0], neighbour[1]] == -1:
            continue
        elif visited[neighbour[0], neighbour[1]] == 0:
            print(f"Connecting tile {current} with {neighbour}")
            connect(maze, tuple(current), tuple(neighbour))
    except IndexError:
        continue
    current = neighbour.copy()
    visited[current[0], current[1]] = 1"""
# Wilson Algorithm
yet = set()
for i in range(maze.shape[0]):
    for j in range(maze.shape[1]):
        if visited[i, j] == 0:
            yet.add((i, j))
#print(maze)
while len(yet) > 0:
    path = []
    current = rng.choices([*yet], k=1)[0]
    visited[current[0], current[1]] = 2
    path.append(current)
    last_direction = ()
    while current in yet:
        direct = direction()
        while last_direction == direct:
            direct = direction()
        try:
            if direct[0] == "V":
                if current[0] + direct[1] in (-1, maze.shape[0]):
                    raise IndexError
                neighbour = (current[0] + direct[1], current[1])
            else:
                if current[1] + direct[1]  in (-1, maze.shape[1]):
                    raise IndexError
                neighbour = (current[0], current[1] + direct[1])
        except IndexError:
            continue
        if visited[neighbour[0], neighbour[1]] == -1:
            continue
        elif visited[neighbour[0], neighbour[1]] == 1:
            path.append(neighbour)
            break
        elif neighbour in path:
            wilson_erase(visited, path, neighbour)
        current = neighbour
        visited[current[0], current[1]] = 2
        if current not in path:
            path.append(current)
    for tile in path:
        visited[tile[0], tile[1]] = 1
        yet.discard(tile)
    while len(path) > 1:
        connect(maze, path[-2], path[-1])
        current = path[-1]
        path.pop()
    path.clear()

base = "0123456789abcdef"
for i in range(maze.shape[0]):
    for j in range(maze.shape[1]):
        try:
            check_tile(maze, (i, j))
            print(base[maze[i][j]], end="")
        except MazeError as e:
            print(e.msg)
    print()
if 0 in visited:
    print("There are still unvisited tiles")

9179793d1397bbb955517d397bbb97b
ec103ea96c29002c17be95005684412
b96aad06d3aeeaa929056bec3bebd2e
c43ec7857c03baeac2a95457a87c3c3
bb857d6bd16c403ad6ead3938015052
aa87d1569697d682d552d0686eebeba
a84512bd4503fbaefffad6d4157852a
ac7beac3d52afc4157fc3957c396d2e
81569454396efffafff942913a857ab
ac7d2957c07d3bfafd52d42aec6b902
a91782d3b8796afafff87bead13aeae
82ed6a96807abc52d3d43abc3e86bc3
aed3bea96a94453e907baaa9052b852
ad12814696c157c16abc006a87c287a
87ac687ba97ad17e92852ed2c796e96
83817ad280107839686d2d56d103d43
ec6c7c7c6eec7ec6d47d47d556ec7d6


## Maze solution

Gonna be a recursive backtracker.

Better: A\*:

```python
# G cost: distance from starting node
# H (heuristic) cost: distance from end node
# h_cost = abs(x_1 - x_2) + abs(y_1 - y_2)
# F cost: G cost + H cost
# open: set of nodes to be evaluated
# closed: set of already evaluated nodes

while True:
    current = "node in open with lowest f_cost"
    remove(current, open)
    add(current, closed)
    
    if current == exit:
        return
    
    for i in neighbours(current):
        if accessible(i) == 0 or i in closed:
            continue

        if f_cost(i) < f_cost(current) or i not in open:
            set_f_cost(i)
            current = parent(i)
            if i not in open:
                add(i, open)
```

## Maze output

In [5]:
txt = ""
with open(conf['OUTPUT_FILE'], 'w') as out:
    print(maze)
    for i in range(size[0]):
        for j in range(size[1]):
            out.write(base[maze[i][j]])
        out.write("\n")

    out.write("\n")
    out.write(f"{entr}          # entry (x,y)\n")
    out.write(f"{exit}          # exit (x,y)\n")
    # out.write(sol)

[[ 9  1  7  9  7  9  3 13  1  3  9  7 11 11 11  9  5  5  5  1  7 13  3  9
   7 11 11 11  9  7 11]
 [14 12  1  0  3 14 10  9  6 12  2  9  0  0  2 12  1  7 11 14  9  5  0  0
   5  6  8  4  4  1  2]
 [11  9  6 10 10 13  0  6 13  3 10 14 14 10 10  9  2  9  0  5  6 11 14 12
   3 11 14 11 13  2 14]
 [12  4  3 14 12  7  8  5  7 12  0  3 11 10 14 10 12  2 10  9  5  4  5  7
  10  8  7 12  3 12  3]
 [11 11  8  5  7 13  6 11 13  1  6 12  4  0  3 10 13  6 14 10 13  3  9  3
   8  0  1  5  0  5  2]
 [10 10  8  7 13  1  5  6  9  6  9  7 13  6  8  2 13  5  5  2 13  0  6  8
   6 14 14 11 14 11 10]
 [10  8  4  5  1  2 11 13  4  5  0  3 15 11 10 14 15 15 15 10 13  6 13  4
   1  5  7  8  5  2 10]
 [10 12  7 11 14 10 12  3 13  5  2 10 15 12  4  1  5  7 15 12  3  9  5  7
  12  3  9  6 13  2 14]
 [ 8  1  5  6  9  4  5  4  3  9  6 14 15 15 15 10 15 15 15  9  4  2  9  1
   3 10  8  5  7 10 11]
 [10 12  7 13  2  9  5  7 12  0  7 13  3 11 15 10 15 13  5  2 13  4  2 10
  14 12  6 11  9  0  2]
 [10  9  1  7  8  2 